# Greedy Optimized Smoke Test

Quick notebook copy that validates the capped class-wise SMOTE path on a stratified subset of the preprocessed CSV cache.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DRIVE_RESULTS_DIR = os.environ.get(
    "SMOKE_RESULTS_DIR",
    str(Path.home() / "Desktop" / "station_greedy" / "results"),
)
SEQ_LENGTH = 10
STRIDE = 10

CSV_USE_BALANCED_PREPROCESSED = True
CSV_SMOTE_FORCE_REBUILD = True
CSV_SMOTE_TARGET_QUANTILE = 0.50
CSV_SMOTE_MAX_MULTIPLIER = 4.0
CSV_SMOTE_MAX_NEW_SAMPLES = 4096
CSV_SMOTE_CONTEXT_MULTIPLIER = 1.25
CSV_SMOTE_K_NEIGHBORS = 5
CSV_SMOTE_RANDOM_STATE = 42

CSV_PREPROCESSED_DIR = f"{DRIVE_RESULTS_DIR}/preprocessed/csv"
CSV_SMOTE_PREPROCESSED_DIR = f"{DRIVE_RESULTS_DIR}/preprocessed/csv_smoke"

print("Project root:", PROJECT_ROOT)
print("Base CSV cache:", CSV_PREPROCESSED_DIR)
print("Smoke CSV cache:", CSV_SMOTE_PREPROCESSED_DIR)


In [ ]:
# ─── Cell: RAM Monitoring & Data Loading ─────────────────────────────────
import gc
import os
import pickle
from collections import Counter

import numpy as np
import psutil
import torch


def get_memory_usage():
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024**3)
    gpu_gb = 0
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / (1024**3)
    return ram_gb, gpu_gb


def log_memory(label=''):
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [RAM {label}] {ram_gb:.2f} GB | [GPU {label}] {gpu_gb:.2f} GB')


def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [Cleanup] RAM: {ram_gb:.2f} GB | GPU: {gpu_gb:.2f} GB')


def print_class_distribution(y, prefix=''):
    counts = Counter(np.asarray(y).ravel().tolist())
    total = sum(counts.values())
    for cls_id, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
        pct = 100.0 * count / max(total, 1)
        print(f'    {prefix}class={cls_id:<3d} count={count:>8,} ({pct:>5.2f}%)')
    return counts


def _load_preprocessed_dataset(preprocessed_dir, dataset_type):
    ready_file = f'{preprocessed_dir}/{dataset_type}_ready'
    train_file = f'{preprocessed_dir}/X_train.npy'
    meta_file = f'{preprocessed_dir}/{dataset_type}_metadata.pkl'

    if not (os.path.exists(ready_file) and os.path.exists(train_file) and os.path.exists(meta_file)):
        return None

    X_train = np.load(f'{preprocessed_dir}/X_train.npy')
    X_val = np.load(f'{preprocessed_dir}/X_val.npy')
    X_test = np.load(f'{preprocessed_dir}/X_test.npy')
    y_train = np.load(f'{preprocessed_dir}/y_train.npy')
    y_val = np.load(f'{preprocessed_dir}/y_val.npy')
    y_test = np.load(f'{preprocessed_dir}/y_test.npy')

    with open(meta_file, 'rb') as f:
        metadata = pickle.load(f)

    data = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': metadata['features'],
        'scaler': metadata['scaler'],
        'label_encoder': metadata['label_encoder'],
        'n_continuous': metadata['n_continuous'],
    }
    print(f'  Preprocessed {dataset_type.upper()} loaded from {preprocessed_dir}: {len(X_train):,} train samples')
    return data


def _save_preprocessed_dataset(preprocessed_dir, dataset_type, data):
    os.makedirs(preprocessed_dir, exist_ok=True)
    np.save(f'{preprocessed_dir}/X_train.npy', data['X_train'])
    np.save(f'{preprocessed_dir}/X_val.npy', data['X_val'])
    np.save(f'{preprocessed_dir}/X_test.npy', data['X_test'])
    np.save(f'{preprocessed_dir}/y_train.npy', data['y_train'])
    np.save(f'{preprocessed_dir}/y_val.npy', data['y_val'])
    np.save(f'{preprocessed_dir}/y_test.npy', data['y_test'])
    with open(f'{preprocessed_dir}/{dataset_type}_metadata.pkl', 'wb') as f:
        pickle.dump({
            'features': data['features'],
            'scaler': data['scaler'],
            'label_encoder': data['label_encoder'],
            'n_continuous': data['n_continuous'],
            'seq_length': SEQ_LENGTH,
            'stride': STRIDE,
        }, f)
    with open(f'{preprocessed_dir}/{dataset_type}_ready', 'w') as f:
        f.write('ready')


def build_smote_augmentation_plan(y_train):
    counts = Counter(np.asarray(y_train).ravel().tolist())
    if len(counts) < 2:
        return counts, 0, []

    ordered = np.array(sorted(counts.values()), dtype=np.int64)
    target_floor = int(np.quantile(ordered, CSV_SMOTE_TARGET_QUANTILE))
    remaining_budget = int(CSV_SMOTE_MAX_NEW_SAMPLES)
    plan = []

    for cls_id, count in sorted(counts.items(), key=lambda item: item[1]):
        capped_target = min(
            int(np.ceil(count * CSV_SMOTE_MAX_MULTIPLIER)),
            target_floor,
        )
        if capped_target <= count:
            continue
        add_count = capped_target - count
        if remaining_budget <= 0:
            break
        if add_count > remaining_budget:
            capped_target = count + remaining_budget
            add_count = remaining_budget
        if add_count <= 0:
            continue
        plan.append((int(cls_id), int(count), int(capped_target), int(add_count)))
        remaining_budget -= add_count

    return counts, target_floor, plan


def _extract_synthetic_rows(original_minority, resampled_minority):
    original_counter = Counter(np.ascontiguousarray(row).tobytes() for row in original_minority)
    synthetic_rows = []
    for row in resampled_minority:
        key = np.ascontiguousarray(row).tobytes()
        if original_counter.get(key, 0):
            original_counter[key] -= 1
        else:
            synthetic_rows.append(row)
    if synthetic_rows:
        return np.asarray(synthetic_rows, dtype=resampled_minority.dtype)
    return np.empty((0,) + original_minority.shape[1:], dtype=original_minority.dtype)


def _generate_classwise_smote_samples(X_train, y_train, class_id, target_count, rng):
    from imblearn.over_sampling import SMOTE

    class_mask = y_train == class_id
    X_minority = X_train[class_mask]
    current_count = len(X_minority)
    if target_count <= current_count:
        return np.empty((0,) + X_train.shape[1:], dtype=X_train.dtype)

    non_class_indices = np.flatnonzero(~class_mask)
    context_size = min(
        len(non_class_indices),
        max(target_count, int(np.ceil(current_count * CSV_SMOTE_CONTEXT_MULTIPLIER))),
    )
    if context_size == 0:
        return np.empty((0,) + X_train.shape[1:], dtype=X_train.dtype)

    if context_size < len(non_class_indices):
        context_indices = rng.choice(non_class_indices, size=context_size, replace=False)
    else:
        context_indices = non_class_indices

    X_context = X_train[context_indices]
    X_work = np.concatenate([X_minority, X_context], axis=0)
    y_work = np.concatenate([
        np.ones(current_count, dtype=np.int64),
        np.zeros(len(X_context), dtype=np.int64),
    ])

    min_class_count = current_count
    if min_class_count < 2:
        return np.empty((0,) + X_train.shape[1:], dtype=X_train.dtype)

    k_neighbors = max(1, min(CSV_SMOTE_K_NEIGHBORS, min_class_count - 1))
    smote = SMOTE(
        sampling_strategy={1: int(target_count)},
        random_state=int(rng.integers(0, 2**31 - 1)),
        k_neighbors=k_neighbors,
    )

    X_resampled, y_resampled = smote.fit_resample(X_work.reshape(len(X_work), -1), y_work)
    X_minority_resampled = X_resampled[y_resampled == 1].reshape(-1, X_train.shape[1], X_train.shape[2])
    synthetic = _extract_synthetic_rows(X_minority, X_minority_resampled)
    needed = target_count - current_count
    if len(synthetic) < needed:
        raise RuntimeError(
            f'SMOTE generated only {len(synthetic)} synthetic rows for class {class_id}, expected {needed}.'
        )
    return synthetic[:needed].astype(X_train.dtype, copy=False)


def apply_smote_to_preprocessed_csv(data, save_dir=None, force_rebuild=False):
    if not CSV_USE_BALANCED_PREPROCESSED:
        return data

    if save_dir and not force_rebuild:
        cached = _load_preprocessed_dataset(save_dir, 'csv')
        if cached is not None:
            print(f'  Using cached balanced CSV from {save_dir}')
            return cached

    try:
        import imblearn  # noqa: F401
    except ImportError:
        print('  imbalanced-learn is not installed; returning the original preprocessed CSV.')
        return data

    X_train = np.asarray(data['X_train'])
    y_train = np.asarray(data['y_train']).ravel()

    print('  Building a capped class-wise SMOTE cache for the CSV training split...')
    before_counts, target_floor, plan = build_smote_augmentation_plan(y_train)
    print(f'  SMOTE target floor (quantile): {target_floor:,}')
    print(f'  SMOTE budget (new samples max): {CSV_SMOTE_MAX_NEW_SAMPLES:,}')
    if not plan:
        print('  No SMOTE augmentation required with the current conservative caps.')
        return data

    total_new = sum(add_count for _, _, _, add_count in plan)
    print(f'  Planned synthetic samples: {total_new:,}')
    for class_id, current_count, target_count, add_count in plan:
        print(
            f'    class {class_id:<3d}: {current_count:>8,} -> {target_count:>8,} '
            f'(adding {add_count:>8,})'
        )

    rng = np.random.default_rng(CSV_SMOTE_RANDOM_STATE)
    synthetic_batches = []
    synthetic_labels = []

    for class_id, current_count, target_count, add_count in plan:
        print(f'  Running SMOTE for class {class_id} on a reduced working set...')
        synthetic = _generate_classwise_smote_samples(X_train, y_train, class_id, target_count, rng)
        if len(synthetic) == 0:
            print(f'    skipped class {class_id}: no synthetic rows produced')
            continue
        synthetic_batches.append(synthetic)
        synthetic_labels.append(np.full(len(synthetic), class_id, dtype=data["y_train"].dtype))
        aggressive_cleanup()

    if not synthetic_batches:
        print('  SMOTE did not produce any synthetic batches; returning the original data.')
        return data

    X_synthetic = np.concatenate(synthetic_batches, axis=0)
    y_synthetic = np.concatenate(synthetic_labels, axis=0)

    n_continuous = data.get('n_continuous')
    if n_continuous is not None and n_continuous < X_synthetic.shape[-1]:
        X_synthetic[:, :, n_continuous:] = np.clip(np.rint(X_synthetic[:, :, n_continuous:]), 0, 1)

    X_balanced = np.concatenate([X_train, X_synthetic], axis=0)
    y_balanced = np.concatenate([y_train.astype(data['y_train'].dtype, copy=False), y_synthetic], axis=0)

    permutation = rng.permutation(len(y_balanced))
    X_balanced = X_balanced[permutation]
    y_balanced = y_balanced[permutation]

    balanced = dict(data)
    balanced['X_train'] = X_balanced.astype(data['X_train'].dtype, copy=False)
    balanced['y_train'] = y_balanced.astype(data['y_train'].dtype, copy=False)

    print('  Train distribution before SMOTE:')
    print_class_distribution(y_train, prefix='before_')
    print('  Train distribution after SMOTE:')
    print_class_distribution(balanced['y_train'], prefix='after_')

    if save_dir:
        _save_preprocessed_dataset(save_dir, 'csv', balanced)
        print(f'  Balanced CSV cache saved to {save_dir}')

    return balanced


print('RAM monitoring utilities loaded.')
log_memory('startup')


def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    import sys
    sys.path.insert(0, '/content/pfe')
    from src.data.preprocessor import IoTDataProcessor

    print('\n' + '=' * 70)
    print('  LOADING CSV DATASET')
    print('=' * 70)

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f'  Features ({n_continuous}): {features[:5]}...')
    print(f'  Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}')
    print(f'  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

    data = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

    if save_dir:
        print(f'  Saving preprocessed CSV to Drive...')
        _save_preprocessed_dataset(save_dir, 'csv', data)
        print(f'  CSV dataset saved to Drive.')

    return apply_smote_to_preprocessed_csv(
        data,
        save_dir=CSV_SMOTE_PREPROCESSED_DIR if CSV_USE_BALANCED_PREPROCESSED else None,
        force_rebuild=CSV_SMOTE_FORCE_REBUILD,
    )


def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    import sys
    sys.path.insert(0, '/content/pfe')
    from src.data.json_preprocessor import JsonIoTDataProcessor

    print('\n' + '=' * 70)
    print('  LOADING JSON DATASET')
    print('=' * 70)

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36

    print(f'  Features ({len(features)}): {features[:5]}...')
    print(f'  Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}')
    print(f'  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

    data = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

    if save_dir:
        print(f'  Saving preprocessed JSON to Drive...')
        _save_preprocessed_dataset(save_dir, 'json', data)
        print(f'  JSON dataset saved to Drive.')

    return data


def load_dataset_from_drive(dataset_type):
    if dataset_type == 'csv' and CSV_USE_BALANCED_PREPROCESSED:
        cached = _load_preprocessed_dataset(CSV_SMOTE_PREPROCESSED_DIR, 'csv')
        if cached is not None and not CSV_SMOTE_FORCE_REBUILD:
            print('  Loading cached balanced CSV from Drive...')
            return cached

    preprocessed_dir = f'{DRIVE_RESULTS_DIR}/preprocessed/{dataset_type}'
    data = _load_preprocessed_dataset(preprocessed_dir, dataset_type)
    if data is not None:
        if dataset_type == 'csv':
            return apply_smote_to_preprocessed_csv(
                data,
                save_dir=CSV_SMOTE_PREPROCESSED_DIR if CSV_USE_BALANCED_PREPROCESSED else None,
                force_rebuild=CSV_SMOTE_FORCE_REBUILD,
            )
        return data

    print(f'  No preprocessed data found. Loading {dataset_type.upper()} fresh...')
    if dataset_type == 'csv':
        return load_and_display_csv_dataset(
            CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
            save_dir=CSV_PREPROCESSED_DIR
        )
    return load_and_display_json_dataset(
        JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
    )


CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'
CSV_SMOTE_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv_smote'
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

print('Data loading functions ready.')


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from src.models.lstm import LSTMClassifier


def stratified_cap_train_split(data, base_cap=24, random_state=42):
    rng = np.random.default_rng(random_state)
    y_train = np.asarray(data['y_train']).ravel()
    keep_indices = []
    cap_schedule = [base_cap, base_cap * 2, base_cap * 3]
    for offset, class_id in enumerate(sorted(np.unique(y_train))):
        class_indices = np.flatnonzero(y_train == class_id)
        rng.shuffle(class_indices)
        class_cap = cap_schedule[offset % len(cap_schedule)]
        keep_indices.extend(class_indices[: min(len(class_indices), class_cap)].tolist())
    keep_indices = np.asarray(sorted(keep_indices))
    capped = dict(data)
    capped['X_train'] = np.asarray(data['X_train'])[keep_indices]
    capped['y_train'] = y_train[keep_indices].astype(data['y_train'].dtype, copy=False)
    return capped


base_data = _load_preprocessed_dataset(CSV_PREPROCESSED_DIR, 'csv')
if base_data is None:
    raise FileNotFoundError(f'Base preprocessed CSV not found at {CSV_PREPROCESSED_DIR}')

smoke_data = stratified_cap_train_split(base_data, base_cap=24, random_state=CSV_SMOTE_RANDOM_STATE)
balanced_smoke = apply_smote_to_preprocessed_csv(
    smoke_data,
    save_dir=CSV_SMOTE_PREPROCESSED_DIR,
    force_rebuild=True,
)

train_limit = min(len(balanced_smoke['X_train']), 256)
dataset = TensorDataset(
    torch.tensor(balanced_smoke['X_train'][:train_limit], dtype=torch.float32),
    torch.tensor(balanced_smoke['y_train'][:train_limit], dtype=torch.long),
)
loader = DataLoader(dataset, batch_size=min(32, len(dataset)), shuffle=True)
x_batch, y_batch = next(iter(loader))

num_classes = len(base_data['label_encoder'].classes_) if 'label_encoder' in base_data else int(np.max(y_batch.numpy())) + 1
model = LSTMClassifier(
    input_size=balanced_smoke['X_train'].shape[-1],
    num_classes=num_classes,
    config_path='config/config.yaml',
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

optimizer.zero_grad(set_to_none=True)
logits = model(x_batch)
loss = criterion(logits, y_batch)
loss.backward()
optimizer.step()

print('Smoke logits shape:', tuple(logits.shape))
print('Smoke loss:', float(loss.detach().cpu()))
print('Smoke test passed.')
